# Credit Card Fraud Detection: Imbalanced Classification with Cost-Based Evaluation
**Dataset:** Kaggle *Credit Card Fraud Detection* (ULB), file `creditcard.csv`: 284,807 transactions, 492 frauds (0.17%).
Columns: `Time`, `V1`-`V28` (anonymised PCA components), `Amount`, `Class` (1 = fraud).

**What makes this project advanced**
1. Extreme class imbalance, so accuracy is meaningless. We use precision-recall metrics.
2. Imbalance handling compared fairly: class weights, oversampling, undersampling.
3. Gradient boosting vs. a Logistic Regression baseline.
4. A **business cost** (a missed fraud costs far more than a false alarm) used to choose the decision threshold.
5. A proper train / validation / test protocol: the threshold is tuned on validation data, and the test set is used once.
6. Model explanation with permutation importance (SHAP optional).

> Every code line has a `#` comment, so this notebook is also the line-by-line explanation.

## 0. Imports and settings

In [ ]:
import numpy as np                                    # arrays and maths
import pandas as pd                                   # DataFrames
import matplotlib.pyplot as plt                       # plotting
import seaborn as sns                                 # nicer plots
import os, warnings                                   # folders and warning control
from sklearn.model_selection import train_test_split  # splitting data
from sklearn.preprocessing import RobustScaler        # scaler that is not distorted by outliers
from sklearn.pipeline import Pipeline                 # chain scaler + model
from sklearn.compose import ColumnTransformer         # scale only chosen columns
from sklearn.dummy import DummyClassifier             # "always predict majority" baseline
from sklearn.linear_model import LogisticRegression   # linear classifier
from sklearn.ensemble import HistGradientBoostingClassifier   # fast gradient boosting (no extra install)
from sklearn.utils import resample                    # random over/under sampling
from sklearn.inspection import permutation_importance # model-agnostic feature importance
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, f1_score, precision_score, recall_score, accuracy_score)

warnings.filterwarnings("ignore", category=RuntimeWarning)   # hide harmless numeric warnings (seen on some Macs)
RANDOM_STATE = 42                                     # fixed seed for reproducibility
sns.set_style("whitegrid")                            # plot style

# Optional libraries: the notebook still runs if they are not installed
try:
    from imblearn.over_sampling import SMOTE          # synthetic minority oversampling
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False                                 # fall back to random oversampling later
try:
    from xgboost import XGBClassifier                 # XGBoost (pip install xgboost)
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    import shap                                       # SHAP explanations (pip install shap)
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
print("SMOTE:", HAS_SMOTE, "| XGBoost:", HAS_XGB, "| SHAP:", HAS_SHAP)   # show what is available

def save(name):                                       # helper: save figure then show it
    os.makedirs("figs", exist_ok=True)                # create folder if missing
    plt.tight_layout()                                # avoid overlapping labels
    plt.savefig(f"figs/{name}.png", dpi=130)          # write PNG
    plt.show()                                        # display

## 1. Load and inspect

In [ ]:
df = pd.read_csv("creditcard.csv")                    # read the Kaggle file (put it next to this notebook)
print("Shape:", df.shape)                             # rows, columns
print("Missing values:", int(df.isnull().sum().sum()))   # total missing cells
print("Duplicate rows:", df.duplicated().sum())       # exact duplicate rows
df = df.drop_duplicates()                             # remove duplicates (the real file has about 1,000)
df.head()                                             # first rows

In [ ]:
counts = df["Class"].value_counts()                   # number of legit (0) and fraud (1)
print(counts)                                         # raw counts
print("Fraud rate: %.4f%%" % (100 * df["Class"].mean()))   # percentage of fraud
print("Accuracy of always predicting 'legit': %.4f" % (1 - df["Class"].mean()))   # the accuracy trap

**Key point:** a model that never flags anything gets about 99.8% accuracy and catches zero fraud. That is why we use **precision, recall, F1, average precision (area under the PR curve) and money-based cost** instead.

## 2. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))         # three panels
sns.countplot(x="Class", data=df, ax=ax[0]); ax[0].set_yscale("log"); ax[0].set_title("Class counts (log scale)")   # imbalance
sns.histplot(np.log1p(df["Amount"]), bins=60, ax=ax[1]); ax[1].set_title("log(1 + Amount)")   # amounts are very skewed
sns.boxplot(x="Class", y=np.log1p(df["Amount"]), data=df, ax=ax[2]); ax[2].set_title("log(1 + Amount) by class")   # fraud vs legit amounts
save("01_class_amount")                               # save + show

In [ ]:
corr_target = df.corr()["Class"].drop("Class").sort_values(key=abs, ascending=False)   # correlation of each feature with fraud
print(corr_target.head(10))                           # strongest signals
corr_target.head(12).sort_values().plot(kind="barh", figsize=(7, 4), title="Top correlations with Class")   # bar chart
save("02_corr_with_class")                            # save + show

In [ ]:
hours = (df["Time"] // 3600) % 24                     # hour of day from seconds since first transaction
rate = df.groupby(hours)["Class"].mean() * 100        # fraud percentage per hour
rate.plot(kind="bar", figsize=(10, 3.5), title="Fraud rate (%) by hour of day (relative to first transaction)")   # bar plot
save("03_fraud_by_hour")                              # save + show

## 3. Train / validation / test split and scaling
Split **60 / 20 / 20**, **stratified** so each part keeps the same fraud rate.
- **Train**: fit models. **Validation**: choose the decision threshold and compare options. **Test**: used once at the end.
- `V1`-`V28` are already PCA outputs, so only `Time` and `Amount` are scaled (`RobustScaler`, which is not distorted by Amount's huge outliers).
- Any resampling (SMOTE etc.) is applied to the **training** part only, never to validation or test.

In [ ]:
X = df.drop(columns=["Class"])                        # features
y = df["Class"]                                       # target: 1 = fraud
X_tmp, X_test, y_tmp, y_test = train_test_split(      # first cut off 20% for the final test
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(    # then split the rest 75/25 -> 60% train, 20% val overall
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RANDOM_STATE)
for n, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:   # confirm sizes and fraud counts
    print(f"{n:5s} rows={len(yy):7d} frauds={int(yy.sum()):4d} rate={100*yy.mean():.3f}%")

prep = ColumnTransformer([("scale", RobustScaler(), ["Time", "Amount"])],   # scale only Time and Amount
                         remainder="passthrough")     # leave V1-V28 untouched

## 4. Evaluation helpers
For fraud we report **Average Precision (AP)** = area under the precision-recall curve, the main metric, plus ROC-AUC and precision / recall / F1 at a threshold.

In [ ]:
def metrics_at(y_true, proba, thr=0.5):               # metrics for one probability threshold
    pred = (proba >= thr).astype(int)                 # turn probabilities into 0/1 decisions
    return {"AP": average_precision_score(y_true, proba),   # threshold-free quality (main metric)
            "ROC_AUC": roc_auc_score(y_true, proba),        # ranking quality (can look too good when data is imbalanced)
            "Precision": precision_score(y_true, pred, zero_division=0),   # of flagged, how many are fraud
            "Recall": recall_score(y_true, pred),                          # of frauds, how many we caught
            "F1": f1_score(y_true, pred)}                                  # balance of precision and recall

val_proba, test_proba = {}, {}                        # store each model's probabilities
rows = []                                             # store validation metrics for the comparison table
def run(name, model, Xtr, ytr):                       # fit a model and record validation scores
    model.fit(Xtr, ytr)                               # train
    val_proba[name] = model.predict_proba(X_val)[:, 1]    # fraud probability on validation
    test_proba[name] = model.predict_proba(X_test)[:, 1]  # and on test (kept for the end, not used for choices)
    rows.append({"Model": name, **metrics_at(y_val, val_proba[name])})   # save validation metrics
    return model

## 5. Baselines: why accuracy fails

In [ ]:
dummy = Pipeline([("prep", prep), ("m", DummyClassifier(strategy="most_frequent"))])   # always predicts legit
dummy.fit(X_train, y_train)                           # fit (learns only the majority class)
pred = dummy.predict(X_val)                           # predictions
print("Dummy accuracy:", round(accuracy_score(y_val, pred), 4), "| fraud recall:", recall_score(y_val, pred))   # high accuracy, zero recall

In [ ]:
lr_plain = run("LogReg (no imbalance fix)",           # plain Logistic Regression baseline
              Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]), X_train, y_train)
pd.DataFrame(rows).round(4)                           # first row of the comparison table

## 6. Handling imbalance (three approaches)
| Method | Idea | Risk |
|---|---|---|
| **Class weights** | Make mistakes on fraud cost more in the loss function | none (no data changed) |
| **Oversampling / SMOTE** | Add copies or synthetic fraud rows to training data | can overfit to the few real frauds |
| **Undersampling** | Remove legit rows | throws away information |

In [ ]:
lr_w = run("LogReg + class_weight",                   # class weights: errors on the rare class count more
           Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))]), X_train, y_train)

Xp = pd.DataFrame(prep.fit_transform(X_train), columns=["Time", "Amount"] + [c for c in X.columns if c not in ["Time", "Amount"]])   # scaled train features
if HAS_SMOTE:                                         # preferred: synthetic oversampling
    Xs, ys = SMOTE(random_state=RANDOM_STATE).fit_resample(Xp, y_train)    # create synthetic fraud rows (TRAIN only)
    label = "LogReg + SMOTE"
else:                                                 # fallback: random oversampling of the fraud rows
    fr = Xp[y_train.values == 1]                      # fraud rows
    fr_up = resample(fr, replace=True, n_samples=int((y_train == 0).sum() * 0.1), random_state=RANDOM_STATE)   # up to 10% of legit count
    Xs = pd.concat([Xp[y_train.values == 0], fr_up]); ys = np.r_[np.zeros((y_train == 0).sum()), np.ones(len(fr_up))]
    label = "LogReg + random oversampling"
print(label, "-> training class counts:", np.bincount(ys.astype(int)))   # check the new balance

In [ ]:
def run_resampled(name, model, Xs, ys):               # fit a model on already-scaled resampled data
    model.fit(Xs, ys)                                 # train on resampled data
    cols = Xp.columns                                 # column names used when the model was fitted
    Xv = pd.DataFrame(prep.transform(X_val), columns=cols); Xt = pd.DataFrame(prep.transform(X_test), columns=cols)   # apply the SAME scaler to val/test
    val_proba[name] = model.predict_proba(Xv)[:, 1]   # validation probabilities
    test_proba[name] = model.predict_proba(Xt)[:, 1]  # test probabilities (not used for choices)
    rows.append({"Model": name, **metrics_at(y_val, val_proba[name])})   # record validation metrics
    return model
run_resampled(label, LogisticRegression(max_iter=2000), Xs, ys)   # Logistic Regression on oversampled data

# Undersampling: keep all frauds + a random sample of legit rows (here 10 legit per fraud)
legit_idx = np.where(y_train.values == 0)[0]          # positions of legit rows
fraud_idx = np.where(y_train.values == 1)[0]          # positions of fraud rows
keep = np.r_[fraud_idx, np.random.RandomState(RANDOM_STATE).choice(legit_idx, size=10 * len(fraud_idx), replace=False)]   # chosen rows
run_resampled("LogReg + undersampling", LogisticRegression(max_iter=2000), Xp.iloc[keep], y_train.values[keep])   # fit
pd.DataFrame(rows).round(4)                           # comparison so far

## 7. Gradient boosting (tree-based, handles non-linear patterns)
Fraud patterns are usually non-linear and interaction-heavy, so boosted trees normally beat Logistic Regression.

In [ ]:
hgb = run("HistGradientBoosting + class_weight",     # sklearn's fast boosting model
          Pipeline([("prep", prep), ("m", HistGradientBoostingClassifier(
              class_weight="balanced", learning_rate=0.1, max_iter=300, random_state=RANDOM_STATE))]), X_train, y_train)
if HAS_XGB:                                           # only if xgboost is installed
    spw = (y_train == 0).sum() / (y_train == 1).sum() # ratio legit/fraud for scale_pos_weight
    xgb = run("XGBoost + scale_pos_weight",
              Pipeline([("prep", prep), ("m", XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
                        random_state=RANDOM_STATE, n_jobs=-1))]), X_train, y_train)
comp = pd.DataFrame(rows).set_index("Model").round(4)   # comparison table (validation set)
comp.sort_values("AP", ascending=False)               # best average precision first

### 7.1 Precision-recall curves (validation)

In [ ]:
plt.figure(figsize=(8, 6))                            # one plot for all models
for name, p in val_proba.items():                     # each model's curve
    pr, rc, _ = precision_recall_curve(y_val, p)      # precision and recall over all thresholds
    plt.plot(rc, pr, label=f"{name} (AP={average_precision_score(y_val, p):.3f})")   # draw it
plt.axhline(y_val.mean(), color="k", linestyle="--", label="Random (fraud rate)")    # chance level = fraud rate
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-recall curves"); plt.legend(fontsize=8)   # labels
save("04_pr_curves")                                  # save + show

## 8. Business-cost threshold tuning
Default threshold 0.5 is arbitrary. Choose the threshold that minimises **total cost** on the validation set:
- **Missed fraud (false negative):** we lose the transaction amount.
- **False alarm (false positive):** a fixed review cost (assumed **$5**; change `FP_COST` for your scenario).
- **Caught fraud / correct pass:** no cost.

In [ ]:
FP_COST = 5.0                                         # assumed cost of reviewing one false alarm (in currency units)
amt_val = X_val["Amount"].values; amt_test = X_test["Amount"].values   # amounts for cost calculation

def total_cost(y_true, proba, thr, amt):              # money lost at one threshold
    pred = proba >= thr                               # flagged transactions
    fn = (y_true.values == 1) & (~pred)               # frauds we missed
    fp = (y_true.values == 0) & pred                  # legit transactions we flagged
    return amt[fn].sum() + FP_COST * fp.sum()         # lost amounts + review costs

thresholds = np.linspace(0.01, 0.99, 99)              # candidate thresholds
best_thr, summary = {}, []                            # chosen thresholds and results
for name, p in val_proba.items():                     # tune each model on VALIDATION data
    costs = [total_cost(y_val, p, t, amt_val) for t in thresholds]   # cost at every threshold
    best_thr[name] = thresholds[int(np.argmin(costs))]               # threshold with lowest cost
print(pd.Series(best_thr).round(2))                   # chosen thresholds

In [ ]:
best_name = comp["AP"].idxmax()                      # model with the best validation AP
costs = [total_cost(y_val, val_proba[best_name], t, amt_val) for t in thresholds]   # cost curve
plt.plot(thresholds, costs); plt.axvline(best_thr[best_name], color="r", linestyle="--", label=f"best = {best_thr[best_name]:.2f}")   # draw
plt.xlabel("Decision threshold"); plt.ylabel("Total cost (validation)"); plt.title(f"Cost vs threshold: {best_name}"); plt.legend()   # labels
save("05_cost_curve")                                 # save + show

## 9. Final evaluation on the untouched test set
Thresholds were chosen on validation, so the test numbers below are an honest estimate.

In [ ]:
no_model_cost = amt_test[(y_test.values == 1)].sum()  # cost if we never flag anything (lose every fraud)
final = []                                            # final table
for name, p in test_proba.items():                    # evaluate every model on TEST
    thr = best_thr[name]                              # threshold chosen on validation
    m = metrics_at(y_test, p, thr)                    # metrics at that threshold
    final.append({"Model": name, "Threshold": round(thr, 2), **{k: round(v, 4) for k, v in m.items()},
                  "Total cost": round(total_cost(y_test, p, thr, amt_test), 1)})   # money lost
final = pd.DataFrame(final).set_index("Model").sort_values("Total cost")   # cheapest first
print("Cost with no fraud system:", round(no_model_cost, 1))   # reference point
final

In [ ]:
best_final = final.index[0]                           # model with the lowest test cost
pred = (test_proba[best_final] >= best_thr[best_final]).astype(int)   # its decisions on test
cm = confusion_matrix(y_test, pred)                   # TN, FP / FN, TP
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])   # draw
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Test confusion matrix: {best_final}")   # labels
save("06_confusion_test")                             # save + show

## 10. Explaining the model

In [ ]:
imp = permutation_importance(hgb, X_val, y_val,      # explain the gradient-boosting pipeline scoring="average_precision",   # drop in AP when a feature is shuffled
                             n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
imp_s = pd.Series(imp.importances_mean, index=X_val.columns).sort_values(ascending=False)   # rank features
print(imp_s.head(10).round(4))                        # top ten
imp_s.head(12).sort_values().plot(kind="barh", figsize=(7, 4), title="Permutation importance (drop in AP)")   # plot
save("07_importance")                                 # save + show

In [ ]:
if HAS_SHAP:                                          # SHAP needs `pip install shap`
    cols = ["Time", "Amount"] + [c for c in X.columns if c not in ["Time", "Amount"]]   # column order after scaling
    Xv_s = pd.DataFrame(prep.transform(X_val.sample(2000, random_state=RANDOM_STATE)), columns=cols)   # scaled sample
    expl = shap.Explainer(hgb.named_steps["m"], Xv_s)  # explainer for the tree model
    shap.plots.beeswarm(expl(Xv_s), show=False)       # global explanation
    save("08_shap")                                   # save + show
else:
    print("SHAP not installed: run `pip install shap` to add a SHAP summary plot.")

## 11. Conclusions (fill in with your own numbers)
1. **Accuracy is useless** here: the dummy model scores ~99.8% and catches no fraud.
2. Compare **AP** across models: imbalance handling and boosting should clearly raise it versus plain Logistic Regression.
3. The **cost-optimal threshold** is usually far from 0.5 and changes total cost noticeably.
4. Report the **final test table**, the confusion matrix and the top features.

**Limitations:** features are anonymised PCA components, so they cannot be explained in business terms; the $5 review cost is an assumption; the data covers two days, so real systems also need drift monitoring and periodic retraining. In production, the time-ordered nature of transactions favours a time-based split.